In [ ]:
import pandas as pd
import numpy as np
import joblib
import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    precision_recall_curve
)
from category_encoders import TargetEncoder
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print("Imports OK")

In [ ]:
df = pd.read_csv('/media/prince/5A4E832F4E83034D/Fraud-Detector-ML/Data/model_v2_feas.csv')

In [ ]:
num_cols_to_fill = ['transaction_amount', 'time_day', 'time_diff_log']
for col in num_cols_to_fill:
    if col in df.columns and df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

cat_cols_to_fill = [col for col in df.columns if col not in num_cols_to_fill]
for col in cat_cols_to_fill:
    df[col] = df[col].astype(str)

In [ ]:
df.info()

In [ ]:
df = df.drop(columns=['transaction_amount', 'time_day'], axis=1)

In [ ]:
v4 = pd.read_csv('/media/prince/5A4E832F4E83034D/Fraud-Detector-ML/Data/model_v4.csv')
df['amt_log'] = v4['amt_log']
df['time_diff_bins'] = pd.cut(df['time_diff_log'], bins=20, include_lowest=True).astype(object)
df['card_type_user_os'] = v4['card_type_user_os']

In [ ]:
label = df['is_fraud'].astype(int)
predictors = df.drop(columns=['is_fraud'])
x_train, x_test, y_train, y_test = train_test_split(
    predictors, label, test_size=0.2, random_state=42, stratify=label
)
y_train = y_train.astype(int)
y_test  = y_test.astype(int)
print(y_train.value_counts(normalize=True))

In [ ]:
num_cols    = ['amt_log', 'time_diff_log']
target_cols = ['purchaser_email_domain']
cat_cols    = [col for col in x_train.columns if col not in num_cols + target_cols]

In [ ]:
cat_pipeline = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
target_pipeline = Pipeline([
    ('target', TargetEncoder(smoothing=10, handle_unknown='value'))
])
preprocessor = ColumnTransformer([
    ('num',    'passthrough',   num_cols),
    ('target', target_pipeline, target_cols),
    ('cat',    cat_pipeline,    cat_cols)
])

neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
print(f"neg={neg:,}  pos={pos:,}  ratio={neg/pos:.1f}")

model = LGBMClassifier(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 7,
    min_child_samples = 20,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    scale_pos_weight  = neg / pos,   # handles imbalance
    random_state      = 42,
    n_jobs            = -1,
    verbose           = -1
)

final_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])
print("Pipeline ready ✓")

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_predict = cross_val_predict(final_pipe, x_train, y_train, cv=cv, n_jobs=-1)
joblib.dump(cv_predict, 'cv_predict_lgbm.pkl')
print("cv_predict saved ✓", cv_predict.shape)

cv_proba = cross_val_predict(final_pipe, x_train, y_train, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
joblib.dump(cv_proba, 'cv_proba_lgbm.pkl')
print("cv_proba saved ✓", cv_proba.shape)

In [ ]:
final_pipe.fit(x_train, y_train)
joblib.dump(final_pipe, 'model_lgbm.pkl')
print("Model saved ✓")

In [ ]:
print(f"Precision : {precision_score(y_train, cv_predict) * 100:.2f}%")
print(f"Recall    : {recall_score(y_train, cv_predict) * 100:.2f}%")
print()
print(classification_report(y_train, cv_predict))

In [ ]:
precision_vals, recall_vals, thresholds = precision_recall_curve(y_train, cv_proba)

plt.figure(figsize=(8, 5))
plt.plot(thresholds, precision_vals[:-1], 'b-', linewidth=2, label='Precision')
plt.plot(thresholds, recall_vals[:-1],   'g-', linewidth=2, label='Recall')
plt.xlabel('Decision Threshold')
plt.ylabel('Score')
plt.title('Precision and Recall vs Decision Threshold')
plt.legend()
plt.grid(True)
plt.show()